# 🎙️ Sutta TTS Training Control Panel
**Version:** 13.0 | **Author:** SuttaPlayer

This notebook manages your Piper TTS training pipeline with **Tmux persistence**, **Smart Checkpoint Pruning**, and **CPU/GPU flexibility**.

## 🆕 What's New in v13.0
- **`--cpu` Mode:** Test training logic on CPU (batch size 2) without GPU quota.
- **`--batch-size`:** Manually override batch size (default: 8 GPU, 2 CPU).
- **`--ckpt-path`:** Resume from any specific checkpoint file.
- **`--max-epochs`:** Stop training after a specific number of epochs (great for testing).
- **Unified Logging:** Training logs are saved to `/content/train.log` for easy tailing.
- **Deprecated Keep-Alive:** Use a persistent notebook cell instead of the Deno keep-alive loop.

## ⚠️ Critical Notes
- **GPU Quota:** Free tier limits are strict. Use `--cpu` for testing logic.
- **Disk Space:** The `--monitor` command runs smart pruning (keeps top 3 MOS + top 3 MEL) to prevent Drive quota issues.
- **Persistence:** `tmux` keeps training running even if the cell disconnects.

This notebook provides the documented commands to manage your Piper TTS training session. It is designed for the **Colab Free Tier** with a focus on:
1. **Persistence:** Using `tmux` to keep training running if a cell disconnects.
2. **Safety:** Smart pruning to prevent `/content` disk crashes.
3. **Recovery:** Easy restore commands for your environment.

---

## ⚠️ Before You Start
- **Runtime:** Ensure you are using a **GPU Runtime** (preferably T4 or A100 if available). Go to `Runtime > Change runtime type > Hardware accelerator: GPU, Python 3 & Runtime version 2025.07`.
- **Drive:** Ensure Google Drive is mounted in the first cell below.
- **Terminal:** This panel assumes you have access to the Colab Terminal (View > Show terminal).

### The Classic Browser Console Stay-Alive Snippet
If you want to keep your Jupyter interface 100% free and editable while background daemons do all the heavy lifting, you can instruct your browser to simulate physical activity.
1. Press F12 (or right-click and select Inspect) inside Brave/Chrome to open the Developer Tools.
2. Click on the Console tab.
3. Paste the following JavaScript code and press Enter:

```js
function KeepAlive() {
  let connectBtn = document.querySelector("#connect") || document.querySelector("colab-connect-button");
  if (connectBtn) {
    console.log("Simulating click on Connect Button...");
    connectBtn.click();
  }
}
setInterval(KeepAlive, 60000); // Triggers every 60 seconds
```

Why this is bulletproof: This lightweight browser script automatically clicks Colab's connection layout every minute. Even if your kernel is completely idle, this simulated browser interaction prevents Google's frontend idle-checker from ever popping up the "Are you still there?" dialog!

## 🚀 Step 1: Environment Setup & Mounting
Run this cell once to mount Drive and verify your environment.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Verify paths exist
paths = [
    "/content/drive/MyDrive/sutta-tts-model-training",
    "/content/drive/MyDrive/piper_training",
    "/content/piper_cache"
]

print("\n🔍 Checking Path Integrity...")
for p in paths:
    if os.path.exists(p):
        print(f"✅ Found: {p}")
    else:
        print(f"⚠️  Missing: {p} (Creating directory...)")
        os.makedirs(p, exist_ok=True)

# Check Deno installation
try:
    !deno --version
    print("✅ Deno is installed and ready.")
except:
    print("❌ Deno not found. Please run the installation cell below.")

print("\n✅ Environment Ready. Proceed to Step 2.")


## Installation (If Deno is missing)

In [ ]:
# Installation cell for Deno (run only if needed)
print("📦 Installing Deno...")
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'
!deno --version
print("✅ Deno installed successfully.")


## 🚀 Step 2: Launch Training (Tmux Background)

**Why Tmux?** If a Colab cell disconnects, the training process stops. `tmux` keeps it running in a separate terminal session. You can detach from it and come back later.

**Action:** Copy the command below into the **Colab Terminal** (View > Show terminal).

### 📋 Copy This Command to Terminal

**Option A: Standard GPU Training (Resume from default)**
```bash
deno run --allow-all /content/sutta-training-manager.ts --train --batch-size 8
```

**Option B: CPU Testing (2 epochs, batch 2)**
```bash
deno run --allow-all /content/sutta-training-manager.ts --train --cpu --max-epochs 2 --batch-size 2
```

**Option C: Resume from Specific Checkpoint**
```bash
deno run --allow-all /content/sutta-training-manager.ts --train --ckpt-path "/content/drive/MyDrive/piper_training/checkpoints/epoch=9112-val_mos=3.9839.ckpt" --batch-size 8
```

**What happens:**
- Creates a `piper_train` tmux session.
- Logs all output to `/content/train.log`.
- Training continues even if you disconnect the cell.

## 🔍 Step 3: Monitor & Prune (Smart Cleanup)

**Action:** Run this in the **Terminal** to sync checkpoints to Drive and remove duplicates.


### 📋 Copy This Command to Terminal
```bash
deno run --allow-all /content/sutta-training-manager.ts --monitor
```

**What it does:**
1.  Syncs new checkpoints from local to Drive.
2.  **Prunes** local and Drive folders, keeping only:
    - Top 3 by `val_mos` (highest is best).
    - Top 3 by `val_mel` (lowest is best).
    - `last.ckpt` (current state).
3.  Prevents Drive quota overflow.

### 📊 View Live Logs (In Terminal)
```bash
# Follow the training log in real-time
tail -f /content/train.log

# Or view the UAT metrics CSV
tail -f /content/drive/MyDrive/piper_training/uat_metrics.csv
```

### 📋 Run Monitor (Sync & Prune)
Paste this in the **Terminal** to sync checkpoints to Drive and prune local disk:
```bash
deno run --allow-all /content/sutta-training-manager.ts --monitor
```
*This keeps your local disk from filling up.*

## 📊 Step 4: Sync UAT Metrics to Google Sheets
Run this cell to push your latest validation metrics to your Drive CSV and optionally to Sheets.

In [ ]:
import os
from google.colab import drive, sheets
import pandas as pd

drive.mount('/content/drive')

# Paths
CSV_PATH = "./uat_metrics.csv"
DRIVE_CSV = "/content/drive/MyDrive/piper_training/uat_metrics.csv"
SHEET_ID = "YOUR_SHEET_ID_HERE"  # ⚠️ REPLACE THIS WITH YOUR SHEET ID

if os.path.exists(CSV_PATH):
    print("📊 Reading UAT metrics...")
    df = pd.read_csv(CSV_PATH)
    df.to_csv(DRIVE_CSV, index=False)
    print(f"✅ Saved to Drive: {DRIVE_CSV}")
    
    if SHEET_ID != "YOUR_SHEET_ID_HERE":
        print("📤 Appending to Google Sheets...")
        data = df.values.tolist()
        try:
            sheets.values_append(
                sheet_id=SHEET_ID,
                range="A1",
                value_input_option="USER_ENTERED",
                body={"values": data}
            )
            print("🎉 Successfully synced to Google Sheets!")
        except Exception as e:
            print(f"❌ Sheets error: {e}")
    else:
        print("⚠️  No Sheet ID provided. Skipping Sheets append.")
else:
    print("⚠️  No UAT CSV found yet. Training must have run at least one validation.")

## 🔄 Step 5: Recovery & Restore
If you need to reset your environment or restore from a backup.

### 📋 Restore Pip Environment
Run this in the **Terminal** if you need to reinstall dependencies:
```bash
deno run --allow-all /content/sutta-training-manager.ts --pip-restore
```

### 📋 Restore Piper Cache
Run this in the **Terminal** to restore the phoneme cache:
```bash
deno run --allow-all /content/sutta-training-manager.ts --cache-restore
```

## 💾 Step 6: Disk Space Check (Safety)
Run this in the **Terminal** to check your current disk usage before training crashes:

In [ ]:
!df -h /content